# Interacting Maps — live demo

Animated view of the Interacting Maps network (Cook et al., IJCNN 2011, or the
energy-based Martel 2019 thesis network). Sole input: `V` (temporal
derivative). `I`, `G`, `F`, `R` are all inferred.

Notebook counterpart of `demo.py`. Each cell is one atomic step of the
pipeline; all computation is imported from the core modules
(`config`, `interacting_maps.*`) — this notebook only wires them together.

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.gridspec import GridSpec
from IPython.display import HTML

from config import DATASET_CONFIGS, THESIS_PARAMS, COOK_PARAMS, ITERS_PER_FRAME, \
    get_dataset_paths
from interacting_maps.data_loader import EventFrameSequence
from interacting_maps.eval_io import load_imu, get_gyro_for_frame
from interacting_maps.network import InteractingMaps
from interacting_maps.network_dissertation import InteractingMapsThesis
from interacting_maps.viz import normalise, flow_to_rgb

## Configuration

In [2]:
DATASET = 'poster_rotation'
USE_THESIS_VERSION = True
USE_IMU = True  # only applies when USE_THESIS_VERSION = True

# Distortion handling: 'undistort_events' | 'C_full'
DIST_MODE = 'C_full'

## Resolve dataset config, paths, and the warm-start rotation

In [3]:
cfg = DATASET_CONFIGS[DATASET]
paths = get_dataset_paths(DATASET)

initial_R = cfg['initial_R']
if initial_R is None:
    t0, dt = cfg['t_start'], cfg['frame_duration']
    initial_R = get_gyro_for_frame(load_imu(paths['imu']), t0, t0 + dt) * dt
    print(f"Auto-initialized R from IMU: {initial_R}")

T_START = cfg['t_start']
FRAME_DURATION = cfg['frame_duration']
N_FRAMES = cfg['n_frames']

Auto-initialized R from IMU: [-0.00174064 -0.00612314  0.05608617]


## Load the event-frame sequence

In [4]:
seq = EventFrameSequence(
    paths['events'], paths['calib'],
    frame_duration=cfg['frame_duration'],
    t_start=cfg['t_start'],
    n_frames=cfg['n_frames'],
    clip_value=10.0,
    sensor_size=cfg.get('sensor_size', None),
    undistort=(DIST_MODE == 'undistort_events'),        
)
frames = list(seq)
calib, H, W = seq.calib, seq.H, seq.W
fx, fy, cx, cy = calib.fx, calib.fy, calib.cx, calib.cy
print(f"Loaded {len(frames)} frames from {DATASET} ({H}x{W})")

Using calib.txt: fx=199.1 fy=198.8 cx=132.2 cy=110.7
Sensor size: 240x180
Principal point offset from center: dx=12.2px, dy=20.7px
Loaded 150 frames from poster_rotation (180x240)


## Resolve distortion handling for the kinematic matrix `C`

In [5]:
if DIST_MODE == 'undistort_events':
    dist_coeffs = None       # events already undistorted -> pinhole C
elif DIST_MODE == 'C_full':
    dist_coeffs = calib.dist  # distortion-aware C (+ Jacobian)
else:
    raise ValueError(f"Unknown DIST_MODE: {DIST_MODE}")

## Build and initialize the network

In [6]:
if USE_THESIS_VERSION:
    net_params = THESIS_PARAMS.copy()
    net = InteractingMapsThesis(H=H, W=W, fx=fx, fy=fy, cx=cx, cy=cy,
                                dist_coeffs=dist_coeffs, **net_params)
    net.initialize_from_rotation(initial_R)
else:
    net_params = COOK_PARAMS.copy()
    net = InteractingMaps(H=H, W=W, fx=fx, fy=fy, cx=cx, cy=cy,
                          dist_coeffs=dist_coeffs, **net_params)
    net.reset(scale=0.5)
    net.R = initial_R.copy()
    net.F = np.einsum('hwij,j->hwi', net._C_mat, initial_R)  # seed OFCE

## Load IMU gyro (thesis + IMU anchor only)

In [7]:
imu_data = load_imu(paths['imu']) if (USE_THESIS_VERSION and USE_IMU) else None

## Figure setup

In [8]:
fig = plt.figure(figsize=(14, 8))
fig.suptitle(
    f'Interacting Maps — {"Thesis (Martel 2019)" if USE_THESIS_VERSION else "Cook et al. 2011"}  '
    f'[{DATASET}]\n'
    'Sole input: V (temporal derivative). All other maps are inferred.',
    fontsize=10,
)

gs = GridSpec(2, 4, figure=fig, hspace=0.40, wspace=0.30)
ax_V = fig.add_subplot(gs[0, 0])
ax_I = fig.add_subplot(gs[0, 1])
ax_G = fig.add_subplot(gs[0, 2])
ax_F = fig.add_subplot(gs[0, 3])
ax_R = fig.add_subplot(gs[1, 0])
ax_res = fig.add_subplot(gs[1, 1:])

blank, blank3 = np.zeros((H, W)), np.zeros((H, W, 3))
im_V = ax_V.imshow(blank, cmap='RdBu', vmin=-1, vmax=1)
im_I = ax_I.imshow(blank, cmap='gray', vmin=0, vmax=1)
im_G = ax_G.imshow(blank, cmap='hot', vmin=0, vmax=1)
im_F = ax_F.imshow(blank3)

for ax, t in [(ax_V, 'V: Input (temporal deriv.)'),
              (ax_I, 'I: Inferred intensity'),
              (ax_G, '|G|: Inferred gradient'),
              (ax_F, 'F: Inferred flow (HSV)')]:
    ax.set_title(t, fontsize=8)
    ax.axis('off')

r_idx = np.arange(3)
bars_R = ax_R.bar(r_idx, [0, 0, 0], 0.6, color=['#4e79a7', '#f28e2b', '#e15759'])
ax_R.set_xticks(r_idx)
ax_R.set_xticklabels(['Rx', 'Ry', 'Rz'])
ax_R.set_title('Estimated camera rotation R (rad/frame)', fontsize=8)
ax_R.axhline(0, color='k', linewidth=0.5)
ax_R.set_ylim(-0.05, 0.05)

res_VFG_hist, res_GI_hist = [], []
line_VFG, = ax_res.semilogy([], [], 'b-', lw=1.5, label='|V + F·G|  (flow constraint)')
line_GI, = ax_res.semilogy([], [], 'r-', lw=1.5, label='|G − ∇I|   (gradient constraint)')
ax_res.set_title('Constraint residuals over frames', fontsize=8)
ax_res.set_xlabel('Frame')
ax_res.legend(fontsize=7)
ax_res.set_xlim(0, N_FRAMES)
ax_res.set_ylim(1e-5, 2)

plt.close(fig)  # don't show the static scaffold; the animation renders it below

## Per-frame update (one step of the network + one redraw)

In [9]:
def update(t):
    if t >= len(frames):
        return im_V, im_I, im_G, im_F, *bars_R, line_VFG, line_GI

    V, _ = frames[t]

    if USE_THESIS_VERSION and USE_IMU:
        t_lo = T_START + t * FRAME_DURATION
        t_hi = t_lo + FRAME_DURATION
        omega_imu = get_gyro_for_frame(imu_data, t_lo, t_hi)
        net.step(V, n_iters=ITERS_PER_FRAME, omega_imu=omega_imu)
    else:
        net.step(V, n_iters=ITERS_PER_FRAME)

    im_V.set_data(V)
    vabs = max(np.percentile(np.abs(V), 95), 1e-3)
    im_V.set_clim(-vabs, vabs)

    im_I.set_data(normalise(net.I[:H, :W]))
    im_G.set_data(normalise(np.linalg.norm(net.G, axis=-1)))
    im_F.set_data(flow_to_rgb(net.F))

    for bar, val in zip(bars_R, net.R):
        bar.set_height(val)
    ax_R.set_ylim(min(-0.05, net.R.min() * 1.5), max(0.05, net.R.max() * 1.5))

    res_VFG_hist.append(net.residual_VFG(V) + 1e-9)
    res_GI_hist.append(net.residual_GI() + 1e-9)
    xs = list(range(len(res_VFG_hist)))
    line_VFG.set_data(xs, res_VFG_hist)
    line_GI.set_data(xs, res_GI_hist)

    sparsity = np.mean(V == 0) * 100
    print(f'Frame {t+1:3d}/{N_FRAMES}  res_VFG={res_VFG_hist[-1]:.4f}  '
          f'res_GI={res_GI_hist[-1]:.5f}  V_sparsity={sparsity:.0f}%  '
          f'|R|={np.linalg.norm(net.R):.3e}')

    return im_V, im_I, im_G, im_F, *bars_R, line_VFG, line_GI

## Run the animation

In [10]:
ani = animation.FuncAnimation(
    fig, update, frames=N_FRAMES, interval=100, blit=False, repeat=False,
)
plt.tight_layout(rect=[0, 0, 1, 0.93])
HTML(ani.to_jshtml())

Frame   1/150  res_VFG=0.0118  res_GI=0.02043  V_sparsity=57%  |R|=1.428e-02
Frame   1/150  res_VFG=0.0103  res_GI=0.01646  V_sparsity=57%  |R|=1.413e-02
Frame   2/150  res_VFG=0.0129  res_GI=0.02117  V_sparsity=56%  |R|=1.483e-02
Frame   3/150  res_VFG=0.0133  res_GI=0.02192  V_sparsity=56%  |R|=1.491e-02
Frame   4/150  res_VFG=0.0130  res_GI=0.02200  V_sparsity=54%  |R|=1.575e-02
Frame   5/150  res_VFG=0.0122  res_GI=0.02183  V_sparsity=53%  |R|=1.705e-02
Frame   6/150  res_VFG=0.0117  res_GI=0.02144  V_sparsity=53%  |R|=1.721e-02
Frame   7/150  res_VFG=0.0120  res_GI=0.02170  V_sparsity=53%  |R|=1.659e-02
Frame   8/150  res_VFG=0.0127  res_GI=0.02212  V_sparsity=52%  |R|=1.560e-02
Frame   9/150  res_VFG=0.0138  res_GI=0.02225  V_sparsity=53%  |R|=1.394e-02
Frame  10/150  res_VFG=0.0149  res_GI=0.02229  V_sparsity=54%  |R|=1.249e-02
Frame  11/150  res_VFG=0.0157  res_GI=0.02257  V_sparsity=55%  |R|=1.201e-02
Frame  12/150  res_VFG=0.0160  res_GI=0.02294  V_sparsity=54%  |R|=1.188e-02

Animation size has reached 21015221 bytes, exceeding the limit of 20971520.0. If you're sure you want a larger animation embedded, set the animation.embed_limit rc parameter to a larger value (in MB). This and further frames will be dropped.


Frame  57/150  res_VFG=0.0231  res_GI=0.01147  V_sparsity=75%  |R|=2.785e-03
Frame  58/150  res_VFG=0.0189  res_GI=0.00553  V_sparsity=82%  |R|=1.379e-03
Frame  59/150  res_VFG=0.0076  res_GI=0.00091  V_sparsity=93%  |R|=2.789e-04
Frame  60/150  res_VFG=0.0079  res_GI=0.00105  V_sparsity=93%  |R|=4.396e-04
Frame  61/150  res_VFG=0.0093  res_GI=0.00145  V_sparsity=92%  |R|=7.249e-04
Frame  62/150  res_VFG=0.0117  res_GI=0.00212  V_sparsity=90%  |R|=1.087e-03
Frame  63/150  res_VFG=0.0119  res_GI=0.00231  V_sparsity=90%  |R|=1.239e-03
Frame  64/150  res_VFG=0.0120  res_GI=0.00215  V_sparsity=90%  |R|=9.085e-04
Frame  65/150  res_VFG=0.0074  res_GI=0.00087  V_sparsity=92%  |R|=2.563e-04
Frame  66/150  res_VFG=0.0063  res_GI=0.00078  V_sparsity=98%  |R|=6.336e-04
Frame  67/150  res_VFG=0.0095  res_GI=0.00194  V_sparsity=95%  |R|=1.006e-03
Frame  68/150  res_VFG=0.0072  res_GI=0.00118  V_sparsity=95%  |R|=9.031e-04
Frame  69/150  res_VFG=0.0078  res_GI=0.00110  V_sparsity=93%  |R|=6.331e-04

<Figure size 640x480 with 0 Axes>

## Final estimated rotation

In [11]:
print('Final estimated rotation R:')
print(f'  R = {net.R}')
print(f'  |R| = {np.linalg.norm(net.R):.4e}')
print(f'  ω = R/dt = {net.R / FRAME_DURATION} rad/s')

Final estimated rotation R:
  R = [-0.00077155  0.00250553 -0.0031691 ]
  |R| = 4.1129e-03
  ω = R/dt = [-0.03857753  0.12527669 -0.15845524] rad/s
